In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/mersad/protBuild")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import json
import torch
from torch.utils.data import DataLoader
from ml.src.training.dataset.protein_dataset import ProteinDataset
from ml.src.models.tokenizer.bpe import BPETokenizer
from ml.src.models.gpt2.gpt2 import GPT2
from ml.src.training.trainer.train import train_model


In [3]:
config = {}
config_path = PROJECT_ROOT / 'ml/src/models/configs/gpt2-config.json'
selected_config = 'small'
with open (config_path, 'r') as f:
    data = json.load(f)
    if selected_config in data:
        config = data[selected_config]
    else:
        raise KeyError(f"Configuration '{selected_config}' not found.")


In [4]:
vocab_path = PROJECT_ROOT /'ml/datasets/converted/sequences/tokenizer/protein_bpe_vocab.json'
merges_path = PROJECT_ROOT /'ml/datasets/converted/sequences/tokenizer/protein_bpe_merges.json'
train_corpus_path = PROJECT_ROOT /'ml/datasets/converted/sequences/protein_seqs_train.txt'
val_corpus_path = PROJECT_ROOT /'ml/datasets/converted/sequences/protein_seqs_val.txt'
info_corpus_path = PROJECT_ROOT /'ml/datasets/converted/sequences/protein_seqs_info.txt'
end_token='<|endofprotein|>'
checkpoint_path=PROJECT_ROOT /'ml/notebooks/checkpoints/'
save_file_name='checkpoint-model.pth'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = BPETokenizer().load_vocab_and_merges(vocab_path, merges_path)


In [5]:
with open(info_corpus_path, 'r') as f:
    lines = f.readlines()
number_of_proteins = int(lines[0])
split_rate = float(lines[1])

number_of_train_data = int(number_of_proteins * split_rate)
number_of_val_data = number_of_proteins - number_of_train_data

In [6]:
train_data = ProteinDataset(corpus_path=train_corpus_path, tokenizer=tokenizer,
                         context_length=config['context_length'], end_token=end_token)
val_data = ProteinDataset(corpus_path=val_corpus_path, tokenizer=tokenizer,
                         context_length=config['context_length'], end_token=end_token)

In [7]:
train_loader = DataLoader(
    train_data,
    batch_size=2,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

val_loader = DataLoader(
    val_data,
    batch_size=2,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

In [8]:
gpt2_model = GPT2(
    emb_dim=config['emb_dim'],
    d_out=config['emb_dim'],
    vocab_size=config['vocab_size'],
    context_length=config['context_length'],
    num_heads=config['n_heads'],
    n_layers=config['n_layers'],
    dropout=0.1,
    qkv_bias=config['qkv_bias']
)

In [9]:
optimizer = torch.optim.AdamW(gpt2_model.parameters(), lr=0.0001, weight_decay=0.1)

In [10]:
train_model(gpt2_model, train_loader, val_loader, number_of_train_data, number_of_val_data,
             optimizer,device,10, 500, 5, checkpoint_path, save_file_name)

Epoch 1/10: 0batch [00:00, ?batch/s, loss=8.662]

Epoch 1 (Step 0):
Train loss 8.537, Val loss 8.625


Epoch 1/10: 500batch [02:48,  3.01batch/s, loss=7.134]

Epoch 1 (Step 500):
Train loss 7.235, Val loss 7.245


Epoch 1/10: 1000batch [05:36,  3.00batch/s, loss=6.990]

Epoch 1 (Step 1000):
Train loss 7.188, Val loss 7.230


Epoch 1/10: 1338batch [07:31,  2.97batch/s, loss=7.137]


KeyboardInterrupt: 